# 08 · Cross-validation and hyperparameter tuning

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Tune preprocessing and model parameters without test leakage; measure uncertainty across folds.


## Step 1 — Separate final evaluation

The test set remains untouched until a candidate is selected.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           weights=[0.75, 0.25], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)


**Check your result:** Training folds, not test rows, will select C.


## Step 2 — Search training folds

The scaler is refitted inside each fold. Stratification keeps class proportions similar.


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
cv = StratifiedKFold(n_splits=4,shuffle=True,random_state=42)
pipeline = make_pipeline(StandardScaler(),LogisticRegression(max_iter=500))
search = GridSearchCV(pipeline,{'logisticregression__C':[0.01,0.1,1.,10.]},cv=cv,scoring='roc_auc',n_jobs=1,return_train_score=True)
search.fit(X_train,y_train)
print('Selected:',search.best_params_,'mean CV AUC:',search.best_score_)
for mean,std,params in zip(search.cv_results_['mean_test_score'],search.cv_results_['std_test_score'],search.cv_results_['params']):
    print(params, round(mean,4), '+/-',round(std,4))


**Check your result:** Four candidate scores and their fold variability are printed.


## Step 3 — Evaluate the selected pipeline

GridSearchCV refits the best pipeline on all training rows.


In [ ]:
auc = roc_auc_score(y_test,search.predict_proba(X_test)[:,1])
assert 0 <= auc <= 1
print('Final held-out AUC:',auc)


**Check your result:** A valid held-out AUC is reported.


## Practice

Explain when GroupKFold or TimeSeriesSplit should replace random stratified folds.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Group by customer to prevent the same person appearing on both sides. For time-ordered prediction, train on past rows and validate on later rows.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
